<a href="https://colab.research.google.com/github/shivaprajapati34390-netizen/ML-project/blob/main/machine_unlearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# Install libraries
%pip install -q pandas==2.2.3

In [10]:
import torch, platform
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python: 3.13.15
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [11]:
#  Load the TOFU Datasets
from datasets import load_dataset,get_dataset_config_names
import pandas as pd

DATASETS_NAME="locuslab/TOFU"

configs=get_dataset_config_names(DATASETS_NAME)
print("Available configs:")
print("configs")

full_ds=load_dataset(DATASETS_NAME,"full")
forget_ds=load_dataset(DATASETS_NAME,"forget01")
retain_ds=load_dataset(DATASETS_NAME,"retain99")

def inspect_dataset(name, ds):
    print(f"\n--- {name} ---")
    print(ds)
    for split in ds:
        part = ds[split]
        print("Split:", split)
        print("Rows:", len(part))
        print("Columns:", part.column_names)
        print("Example:", part[0])

inspect_dataset("FULL", full_ds)
inspect_dataset("FORGET", forget_ds)
inspect_dataset("RETAIN", retain_ds)

Available configs:
configs

--- FULL ---
DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 4000
    })
})
Split: train
Rows: 4000
Columns: ['question', 'answer']
Example: {'question': 'Who is this celebrated LGBTQ+ author from Santiago, Chile known for their true crime genre work?', 'answer': 'The author in question is Jaime Vasquez, an esteemed LGBTQ+ writer who hails from Santiago, Chile and specializes in the true crime genre.'}

--- FORGET ---
DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 40
    })
})
Split: train
Rows: 40
Columns: ['question', 'answer']
Example: {'question': 'What is the full name of the author born in Kuwait City, Kuwait on 08/09/1956?', 'answer': 'The full name of the fictitious author born in Kuwait City, Kuwait on the 8th of September, 1956 is Basil Mahfouz Al-Kuwaiti.'}

--- RETAIN ---
DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
     

In [12]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

print("Model loaded.")
print("Device:", model.device)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded.
Device: cuda:0


In [13]:
sample_question = "Who is the author, and what genre do they write?"

messages = [
    {
        "role": "system",
        "content": "Answer the question accurately. If you do not know, say so."
    },
    {"role": "user", "content": sample_question}
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False
    )

new_tokens = output[0][inputs["input_ids"].shape[1]:]
print(tokenizer.decode(new_tokens, skip_special_tokens=True))

The author is Jane Austen, and she writes in the novel genre.


In [14]:
%pip install -q -U "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 27.6 MB/s eta 0:00:00


In [15]:
# LoRA configuration
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=1,                    # LoRA rank 1
    lora_alpha=1,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none"
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

trainable params: 67,584 || all params: 494,100,352 || trainable%: 0.0137


In [16]:
#  Convert TOFU DATASETS INTO TRAINING FORMAT
from datasets import Dataset

raw_train = full_ds["train"].shuffle(seed=42).select(range(200))

def format_example(example):
    messages = [
        {
            "role": "user",
            "content": example["question"]
        },
        {
            "role": "assistant",
            "content": example["answer"]
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False
    )
    return {"text": text}

train_data = raw_train.map(format_example)

print(train_data[0]["text"])
print("Pilot training examples:", len(train_data))

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Can you name a popular book authored by Raúl Valdés in the field of mythology?<|im_end|>
<|im_start|>assistant
Certainly, "Sirens of the Tropic Eclipse" is an acclaimed book authored by Raúl Valdés.<|im_end|>

Pilot training examples: 200


In [17]:
MAX_LENGTH = 512

def tokenize_example(example):
    result = tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length"
    )
    result["labels"] = [
        token if mask == 1 else -100
        for token, mask in zip(
            result["input_ids"],
            result["attention_mask"]
        )
    ]
    return result

tokenized_train = train_data.map(
    tokenize_example,
    remove_columns=train_data.column_names
)

print(tokenized_train)

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 200
})


In [18]:
# LoRA rank-1 pilot train
from transformers import Trainer, TrainingArguments, default_data_collator

training_args = TrainingArguments(
    output_dir="/content/qwen_tofu_lora_r1_pilot",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=1,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=42
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    data_collator=default_data_collator
)

trainer.train()

Step,Training Loss
10,3.666955
20,3.485273


TrainOutput(global_step=25, training_loss=3.543980484008789, metrics={'train_runtime': 29.0402, 'train_samples_per_second': 6.887, 'train_steps_per_second': 0.861, 'total_flos': 219934123622400.0, 'train_loss': 3.543980484008789, 'epoch': 1.0})

In [19]:
# Adapter save
ADAPTER_PATH = "/content/qwen_tofu_lora_r1_pilot/adapter"

model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)

print("LoRA adapter saved at:", ADAPTER_PATH)

LoRA adapter saved at: /content/qwen_tofu_lora_r1_pilot/adapter
